# 🤖 AI Engineering Fundamentals — Lezione 3
## Notebook Gruppo B

**ITS Novitas 4.0 | Martedì 26/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "B"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
!pip install anthropic -q
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo B: Conversation History & Multi-turno

Esplorate come funziona la memory del chatbot attraverso la history,
e confrontate le tre strategie di gestione.

---
### Esercizio 1 — Con vs senza history *(guidato)*

La differenza più importante della lezione.
Stessa conversazione, due approcci: solo l'ultimo messaggio vs tutta la history.

In [ ]:
# Esercizio 1 — il chatbot che dimentica vs quello che ricorda

domande = [
    "Mi chiamo Luca e studio AI Engineering a Sassari.",
    "Qual è la capitale della Sardegna?",
    "Come mi chiamo e cosa studio?",  # ← il test della memoria
]

# ── CHATBOT SENZA HISTORY ──────────────────────────────────────────
print("=" * 55)
print("CHATBOT SENZA HISTORY (manda solo l'ultimo messaggio)")
print("=" * 55)

for domanda in domande:
    # TODO: chiamate l'API con SOLO l'ultimo messaggio
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content": ___}]  # solo l'ultimo
    )
    print(f"👤 {domanda}")
    print(f"🤖 {risposta.content[0].text}\n")

print()

# ── CHATBOT CON HISTORY ────────────────────────────────────────────
print("=" * 55)
print("CHATBOT CON HISTORY (manda tutta la lista)")
print("=" * 55)

history = []
for domanda in domande:
    history.append({"role": "user", "content": domanda})

    # TODO: chiamate l'API con TUTTA la history
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=___  # tutta la lista
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    print(f"👤 {domanda}")
    print(f"🤖 {testo}\n")

# Osservazione: cosa risponde alla terza domanda nei due casi?
# ...

---
### Esercizio 2 — Truncation vs Sliding Window *(guidato)*

Fate una conversazione lunga (8 turni) con le due strategie.
Verificate cosa ricorda e cosa dimentica il chatbot in ogni caso.

In [ ]:
# Esercizio 2 — truncation vs sliding window

MAX = 3  # manteniamo solo 3 turni per rendere visibile l'effetto

def chat_truncation(messaggio, history):
    """Tronca la history modificando direttamente la lista."""
    history.append({"role": "user", "content": messaggio})

    # TODO: se len(history) > MAX*2, tronca tenendo solo gli ultimi MAX*2
    ___

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

def chat_sliding(messaggio, history):
    """Usa sliding window: la history completa rimane, manda solo una finestra."""
    history.append({"role": "user", "content": messaggio})

    # TODO: crea messaggi_da_inviare = history[-MAX*2:]
    messaggi_da_inviare = ___

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=messaggi_da_inviare
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test con 6 turni
turni = [
    "Info 1: Mi chiamo Marco.",
    "Info 2: Lavoro a WiData.",
    "Info 3: Il mio sensore preferito è XS200.",
    "Domanda generica: cos'è il RAG?",
    "Altra domanda generica: cos'è lo streaming?",
    "Test memoria: come mi chiamo, dove lavoro e qual è il mio sensore preferito?",
]

hist_trunc = []
hist_slid  = []

print(f"{'Turno':<8} {'Truncation (history)':<25} {'Sliding (history)':<25}")
print("-" * 60)

for i, turno in enumerate(turni):
    r_t = chat_truncation(turno, hist_trunc)
    r_s = chat_sliding(turno, hist_slid)
    print(f"T{i+1:<7} len={len(hist_trunc):<20} len={len(hist_slid)}")

print("\n--- Risposte al test memoria ---")
print(f"Truncation: {r_t[:150]}")
print(f"Sliding:    {r_s[:150]}")

# Le due strategie danno risultati diversi? Quale ricorda di più?
# ...

---
### Esercizio 3 — Summarization *(libero)*

Implementate la strategia più avanzata: quando la history
supera una soglia, chiedete a Claude di riassumerla
e usate il riassunto come contesto compresso.

In [ ]:
# Esercizio 3 — summarization

SOGLIA_TURNI = 4  # quando superare questa soglia, riassumi

def riassumi_history(history):
    """Chiede a Claude di riassumere la conversazione in max 100 token."""
    testo_history = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in history
    )
    # TODO: chiamate Claude per riassumere la conversazione
    # Il riassunto deve essere max 100 token e mantenere i punti essenziali
    riassunto = ___
    return riassunto

def chat_con_summarization(messaggio, history, riassunto=None):
    """Chatbot con summarization quando la history è troppo lunga."""
    history.append({"role": "user", "content": messaggio})

    # Costruisci il contesto: riassunto (se presente) + ultimi messaggi
    if riassunto and len(history) > SOGLIA_TURNI * 2:
        # TODO: usa il riassunto come system prompt aggiuntivo
        system = f"Contesto della conversazione precedente: {riassunto}"
        messaggi_da_inviare = history[-4:]  # solo gli ultimi 2 turni
    else:
        system = None
        messaggi_da_inviare = history

    # TODO: chiamate l'API con system e messaggi_da_inviare
    pass

# Test: create una conversazione lunga, riassumete, continuate
# Verificate che dopo il riassunto il chatbot ricordi ancora le info iniziali
# ...

---
### Esercizio 4 — La strategia giusta per WiData *(libero)*

Un cliente di WiData fa mediamente 15 domande per sessione.
Ogni risposta è lunga circa 100 parole.

Quale strategia di gestione della history consigliate?
Implementatela e motivate la scelta con numeri concreti.

In [ ]:
# Esercizio 4 — la strategia giusta per WiData

# Simulate una sessione tipica WiData: 15 domande sui prodotti
domande_widata = [
    "Quali sensori offrite per ambienti esterni?",
    "Il sensore XS200 funziona con LoRaWAN?",
    "Qual è la durata della batteria?",
    "Come si installa il gateway GW500?",
    "La piattaforma Xplore ha le API REST?",
    # ... aggiungete altre 10 domande
]

# TODO: implementate la strategia che ritenete migliore
# Misurate: token totali, costo, qualità delle risposte

# Confronto finale:
# Strategia scelta: ___
# Motivazione: ___
# Token risparmiati vs nessuna gestione: ___
# Costo per 1000 sessioni/mese: ___

---
## 📊 Preparate la presentazione (5 slide)

1. **Con vs senza history** — la differenza mostrata con i vostri risultati
2. **Il pattern corretto** — i 3 passi: aggiungi user, manda tutto, aggiungi assistant
3. **Truncation vs Sliding Window** — quando usa cosa, con i vostri dati
4. **Summarization** — come funziona e quando conviene
5. **La vostra raccomandazione per WiData** — con motivazione numerica

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*